# Implement Vision Transformer + MAE Pretraining - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google, Apple, Tesla, Waymo

---

### Problem Statement

The **Vision Transformer (ViT)** treats images as sequences of patches and processes them with a standard Transformer encoder. **Masked Autoencoder (MAE)** pretraining randomly masks a high percentage (75%) of patches, feeds only the visible patches through the encoder, and uses a lightweight decoder to reconstruct the masked patches.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate synthetic image data
torch.manual_seed(42)
np.random.seed(42)

def generate_synthetic_images(n_samples, img_size=32, channels=3):
    images = torch.zeros(n_samples, channels, img_size, img_size)
    y = torch.linspace(-1, 1, img_size)
    x = torch.linspace(-1, 1, img_size)
    yy, xx = torch.meshgrid(y, x, indexing='ij')

    for i in range(n_samples):
        pattern = i % 4
        if pattern == 0:
            for c in range(channels):
                freq = np.random.uniform(1, 3)
                phase = np.random.uniform(0, 2 * np.pi)
                images[i, c] = torch.sin(freq * xx + phase)
        elif pattern == 1:
            for c in range(channels):
                freq = np.random.uniform(1, 3)
                phase = np.random.uniform(0, 2 * np.pi)
                images[i, c] = torch.sin(freq * yy + phase)
        elif pattern == 2:
            cx, cy = np.random.uniform(-0.3, 0.3, 2)
            r = torch.sqrt((xx - cx)**2 + (yy - cy)**2)
            for c in range(channels):
                freq = np.random.uniform(2, 5)
                images[i, c] = torch.sin(freq * r)
        else:
            freq = np.random.choice([2, 4, 8])
            checker = torch.sign(torch.sin(freq * np.pi * xx) * torch.sin(freq * np.pi * yy))
            for c in range(channels):
                images[i, c] = checker * np.random.choice([-1, 1])

    images = (images - images.min()) / (images.max() - images.min() + 1e-8)
    return images


img_size = 32
patch_size = 8
channels = 3
n_train = 2000

train_images = generate_synthetic_images(n_train, img_size, channels)
print(f'Training images shape: {train_images.shape}')

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(train_images[i].permute(1, 2, 0).numpy())
    axes[i].set_title(f'Sample {i}')
    axes[i].axis('off')
plt.suptitle('Synthetic Training Images')
plt.tight_layout()
plt.show()

In [ ]:
# --- Part 1: Patch Embedding ---

class PatchEmbedding(nn.Module):
    """
    Split image into non-overlapping patches and project to embedding dimension.
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # Conv2d efficiently extracts and projects all patches
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """
        Args:
            x (Tensor): Images, shape (B, C, H, W).
        Returns:
            Tensor: Patch embeddings, shape (B, n_patches, embed_dim).
        """
        # (B, embed_dim, H/P, W/P) -> (B, embed_dim, n_patches) -> (B, n_patches, embed_dim)
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)  # (B, embed_dim, n_patches)
        x = x.transpose(1, 2)  # (B, n_patches, embed_dim)
        return x


pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=128)
test_img = torch.randn(2, 3, 32, 32)
test_patches = pe(test_img)
print(f'PatchEmbedding: {test_img.shape} -> {test_patches.shape}')
print(f'Number of patches: {pe.n_patches}')

In [ ]:
# --- Part 2: Vision Transformer ---

class ViT(nn.Module):
    """
    Vision Transformer: patch embedding + [CLS] token + positional embedding + transformer encoder.
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128,
                 n_heads=4, n_layers=4, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        # Positional embeddings for CLS + all patches
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, return_all_tokens=False):
        B = x.shape[0]

        # Patch embeddings
        patches = self.patch_embed(x)  # (B, N, D)

        # Prepend [CLS] token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x = torch.cat([cls_tokens, patches], dim=1)  # (B, 1+N, D)

        # Add positional embeddings
        x = x + self.pos_embed

        # Transformer encoder
        x = self.encoder(x)
        x = self.norm(x)

        if return_all_tokens:
            return x
        else:
            return x[:, 0]  # CLS token only


vit = ViT(img_size=32, patch_size=8, in_channels=3, embed_dim=128, n_heads=4, n_layers=4)
test_out = vit(test_img)
print(f'ViT output (CLS only): {test_out.shape}')
test_all = vit(test_img, return_all_tokens=True)
print(f'ViT output (all tokens): {test_all.shape}')

In [ ]:
# --- Part 3: Masked Autoencoder (MAE) ---

class MAE(nn.Module):
    """
    Masked Autoencoder for self-supervised ViT pretraining.
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3,
                 encoder_embed_dim=128, encoder_heads=4, encoder_layers=4,
                 decoder_embed_dim=64, decoder_heads=4, decoder_layers=1,
                 mask_ratio=0.75):
        super().__init__()
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        n_patches = (img_size // patch_size) ** 2
        self.n_patches = n_patches
        patch_dim = patch_size * patch_size * in_channels

        # Encoder
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, encoder_embed_dim)
        self.encoder_pos_embed = nn.Parameter(torch.randn(1, n_patches, encoder_embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=encoder_embed_dim, nhead=encoder_heads,
            dim_feedforward=encoder_embed_dim * 4,
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_layers)
        self.encoder_norm = nn.LayerNorm(encoder_embed_dim)

        # Decoder
        self.decoder_embed = nn.Linear(encoder_embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.randn(1, 1, decoder_embed_dim) * 0.02)
        self.decoder_pos_embed = nn.Parameter(torch.randn(1, n_patches, decoder_embed_dim) * 0.02)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_embed_dim, nhead=decoder_heads,
            dim_feedforward=decoder_embed_dim * 4,
            batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_layers)
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_dim)

    def random_masking(self, x):
        """
        Randomly mask patches.

        Args:
            x (Tensor): Patch embeddings, shape (B, N, D).
        Returns:
            x_visible: Visible patches, shape (B, N_visible, D).
            mask: Binary mask, shape (B, N). 1 = masked.
            ids_restore: Indices to unshuffle, shape (B, N).
        """
        B, N, D = x.shape
        n_keep = int(N * (1 - self.mask_ratio))

        # Random noise for each patch
        noise = torch.rand(B, N, device=x.device)

        # Sort noise to get shuffled indices
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        # Keep first n_keep patches (lowest noise values)
        ids_keep = ids_shuffle[:, :n_keep]
        x_visible = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

        # Binary mask: 0 = visible, 1 = masked
        mask = torch.ones(B, N, device=x.device)
        mask[:, :n_keep] = 0
        # Unshuffle to get mask in original order
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_visible, mask, ids_restore

    def patchify(self, imgs):
        """Convert images to patches for loss computation."""
        B, C, H, W = imgs.shape
        p = self.patch_size
        h = w = H // p
        patches = imgs.reshape(B, C, h, p, w, p)
        patches = patches.permute(0, 2, 4, 3, 5, 1)  # B, h, w, p, p, C
        patches = patches.reshape(B, h * w, -1)  # B, N, patch_dim
        return patches

    def forward(self, images):
        """
        Args:
            images (Tensor): Input images, shape (B, C, H, W).
        Returns:
            loss, pred, mask
        """
        B = images.shape[0]

        # Step 1: Patch embedding + positional embedding
        x = self.patch_embed(images)  # (B, N, encoder_embed_dim)
        x = x + self.encoder_pos_embed

        # Step 2: Random masking
        x_visible, mask, ids_restore = self.random_masking(x)

        # Step 3: Encode visible patches only
        x_encoded = self.encoder(x_visible)
        x_encoded = self.encoder_norm(x_encoded)

        # Step 4: Decoder input - project + insert mask tokens
        x_dec = self.decoder_embed(x_encoded)  # (B, N_visible, decoder_embed_dim)

        # Create mask tokens for masked positions
        n_masked = self.n_patches - x_dec.shape[1]
        mask_tokens = self.mask_token.expand(B, n_masked, -1)

        # Concatenate visible + mask tokens
        x_full = torch.cat([x_dec, mask_tokens], dim=1)  # (B, N, decoder_embed_dim)

        # Unshuffle to restore original patch order
        x_full = torch.gather(
            x_full, dim=1,
            index=ids_restore.unsqueeze(-1).expand(-1, -1, x_full.shape[-1])
        )

        # Step 5: Add decoder positional embeddings and decode
        x_full = x_full + self.decoder_pos_embed
        x_decoded = self.decoder(x_full)
        x_decoded = self.decoder_norm(x_decoded)

        # Step 6: Predict pixel values
        pred = self.decoder_pred(x_decoded)  # (B, N, patch_dim)

        # Step 7: Compute loss on masked patches only
        target = self.patchify(images)  # (B, N, patch_dim)
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)  # per-patch loss, (B, N)
        loss = (loss * mask).sum() / mask.sum()  # mean loss on masked patches

        return loss, pred, mask


mae = MAE(img_size=32, patch_size=8, in_channels=3,
          encoder_embed_dim=128, encoder_heads=4, encoder_layers=4,
          decoder_embed_dim=64, decoder_heads=4, decoder_layers=1)
print(f'MAE parameters: {sum(p.numel() for p in mae.parameters()):,}')

test_loss, test_pred, test_mask = mae(train_images[:4])
print(f'Test loss: {test_loss.item():.4f}')
print(f'Predictions shape: {test_pred.shape}')
print(f'Mask shape: {test_mask.shape}, masked ratio: {test_mask.float().mean():.2f}')

In [ ]:
# --- Part 4: Training ---

optimizer = torch.optim.AdamW(mae.parameters(), lr=1e-3, weight_decay=0.05)
n_epochs = 100
batch_size = 64
losses = []

mae.train()
for epoch in range(n_epochs):
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_train, batch_size):
        imgs = train_images[perm[i:i+batch_size]]

        loss, pred, mask = mae(imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('MAE Training Loss')
plt.show()

In [ ]:
# --- Validation ---

mae.eval()

def unpatchify(pred, patch_size=8, img_size=32, channels=3):
    h = w = img_size // patch_size
    pred = pred.reshape(-1, h, w, patch_size, patch_size, channels)
    pred = pred.permute(0, 5, 1, 3, 2, 4)
    pred = pred.reshape(-1, channels, img_size, img_size)
    return pred


with torch.no_grad():
    test_imgs = train_images[:4]
    loss, pred, mask = mae(test_imgs)

    reconstructed = unpatchify(pred, patch_size, img_size, channels)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i in range(4):
        axes[0, i].imshow(test_imgs[i].permute(1, 2, 0).numpy().clip(0, 1))
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')

        masked_img = test_imgs[i].clone()
        patch_mask = mask[i].reshape(img_size // patch_size, img_size // patch_size)
        for pi in range(img_size // patch_size):
            for pj in range(img_size // patch_size):
                if patch_mask[pi, pj] == 1:
                    masked_img[:, pi*patch_size:(pi+1)*patch_size,
                               pj*patch_size:(pj+1)*patch_size] = 0.5
        axes[1, i].imshow(masked_img.permute(1, 2, 0).numpy().clip(0, 1))
        axes[1, i].set_title('Masked Input')
        axes[1, i].axis('off')

        axes[2, i].imshow(reconstructed[i].permute(1, 2, 0).numpy().clip(0, 1))
        axes[2, i].set_title('Reconstructed')
        axes[2, i].axis('off')

    plt.suptitle('MAE Reconstruction Results')
    plt.tight_layout()
    plt.show()

print('=== Validation ===')
print(f'Final training loss: {losses[-1]:.6f}')
assert losses[-1] < losses[0], 'Reconstruction loss should decrease!'
print('PASSED: Loss decreased during training')

with torch.no_grad():
    patch_emb = mae.patch_embed(test_imgs)
    pos_emb = patch_emb + mae.encoder_pos_embed
    encoded = mae.encoder(pos_emb)
    encoded = mae.encoder_norm(encoded)

    repr_0 = encoded[0].mean(0)
    repr_1 = encoded[1].mean(0)
    repr_2 = encoded[2].mean(0)

    sim_01 = F.cosine_similarity(repr_0.unsqueeze(0), repr_1.unsqueeze(0)).item()
    sim_02 = F.cosine_similarity(repr_0.unsqueeze(0), repr_2.unsqueeze(0)).item()
    print(f'Cosine similarity (img0 vs img1): {sim_01:.4f}')
    print(f'Cosine similarity (img0 vs img2): {sim_02:.4f}')
    print('PASSED: Encoder produces varied representations')

print('\nAll validations passed!')